# Transfer Learning

In [1]:
# ── Install ──────────────────────────────────────────────────
#%pip install tabpfn

In [2]:
import sys
import os

project_root = os.path.abspath("..")
sys.path.append(project_root)

from helpers     import *
from split_data  import *
from preprocessing import *

In [3]:
import numpy as np
from sklearn.base import BaseEstimator, ClassifierMixin
from tabpfn import TabPFNClassifier

# ── Wrapper ───────────────────────────────────────────────────
# TabPFN يدعم max 1000 train sample — الـ wrapper يتعامل معها تلقائياً
class TabPFNWrapper(BaseEstimator, ClassifierMixin):
    def __init__(self, n_estimators=32, subsample=1000, random_state=42):
        self.n_estimators  = n_estimators
        self.subsample     = subsample
        self.random_state  = random_state

    def fit(self, X, y):
        self.classes_ = np.unique(y)

        # Subsample إذا كانت البيانات أكثر من 1000
        if len(y) > self.subsample:
            rng = np.random.RandomState(self.random_state)
            idx = rng.choice(len(y), size=self.subsample, replace=False)
            X, y = X[idx] if hasattr(X, '__getitem__') else X.iloc[idx],                    y.iloc[idx] if hasattr(y, 'iloc') else y[idx]

        self.model_ = TabPFNClassifier(
            n_estimators=self.n_estimators,
            ignore_pretraining_limits=True,   # يتجاوز حد الـ 1000
        )
        self.model_.fit(X, y)
        return self

    def predict(self, X):
        return self.model_.predict(X)

    def predict_proba(self, X):
        return self.model_.predict_proba(X)

print("✅ TabPFNWrapper defined")

✅ TabPFNWrapper defined


In [ ]:
from sklearn.model_selection import cross_validate

pipeline = build_pipeline(TabPFNWrapper(n_estimators=32, random_state=42))

results = cross_validate(
    pipeline, X, y,
    cv=cv,
    scoring=SCORING,
    return_train_score=True,
    n_jobs=1    # TabPFN لا يدعم parallel — خليها 1
)

print("=== TabPFN (Transfer Learning) Results ===")
print(f"Train Accuracy : {results['train_accuracy'].mean():.4f}")
print(f"Val   Accuracy : {results['test_accuracy'].mean():.4f}")
print(f"Val   F1 (w)   : {results['test_f1_weighted'].mean():.4f}")
print(f"Val   F1 (mac) : {results['test_f1_macro'].mean():.4f}")

ValueError: 
All the 5 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
5 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\EbtekarEb\miniconda3\Lib\site-packages\sklearn\model_selection\_validation.py", line 833, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\EbtekarEb\miniconda3\Lib\site-packages\sklearn\base.py", line 1336, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "c:\Users\EbtekarEb\miniconda3\Lib\site-packages\imblearn\pipeline.py", line 522, in fit
    self._final_estimator.fit(Xt, yt, **last_step_params["fit"])
    ~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\EbtekarEb\AppData\Local\Temp\ipyker